# Common vs Independent Initialisation

Reproduces Paper 1 Figure 1 checking loss interpolation between models trained from shared and independent initialisations.

In [1]:
import os
import copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from src.utils import seed_everything, load_mnist
from src.models import TwoNN

seed_everything(42)
os.makedirs('results', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
(train_x, train_y), (test_x, test_y) = load_mnist('data')

Helper functions to train on a 600-example subset and interpolate between two models.

In [2]:
subset_a_x, subset_a_y = train_x[:600], train_y[:600]
subset_b_x, subset_b_y = train_x[600:1200], train_y[600:1200]

def train_subset(init_model, x, y, epochs=15, lr=0.05):
    m = copy.deepcopy(init_model).to(device)
    m.train()
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        perm = torch.randperm(len(y))
        for i in range(0, len(y), 50):
            idx = perm[i:i+50]
            bx, by = x[idx].to(device), y[idx].to(device)
            opt.zero_grad()
            crit(m(bx), by).backward()
            opt.step()
    return m

def eval_loss(model, x, y):
    model.eval()
    crit = nn.CrossEntropyLoss(reduction='mean')
    with torch.no_grad():
        out = model(x.to(device))
        return crit(out, y.to(device)).item()

def interpolate(model_a, model_b, theta):
    interp = copy.deepcopy(model_a)
    dict_a = model_a.state_dict()
    dict_b = model_b.state_dict()
    interp_dict = {}
    for k in dict_a:
        interp_dict[k] = theta * dict_a[k] + (1.0 - theta) * dict_b[k]
    interp.load_state_dict(interp_dict)
    return interp

Train with common initialisation vs independent initialisations and interpolate weights.

In [3]:
seed_everything(100)
init_shared = TwoNN()
model_a_shared = train_subset(init_shared, subset_a_x, subset_a_y)
model_b_shared = train_subset(init_shared, subset_b_x, subset_b_y)

seed_everything(200)
init_indep_a = TwoNN()
seed_everything(300)
init_indep_b = TwoNN()
model_a_indep = train_subset(init_indep_a, subset_a_x, subset_a_y)
model_b_indep = train_subset(init_indep_b, subset_b_x, subset_b_y)

thetas = np.linspace(-0.2, 1.2, 21)
records = []
eval_subset_x, eval_subset_y = train_x[:3000], train_y[:3000]

for th in thetas:
    m_shared = interpolate(model_a_shared, model_b_shared, th)
    loss_s = eval_loss(m_shared, eval_subset_x, eval_subset_y)
    records.append({'mode': 'Shared initialisation', 'theta': float(th), 'loss': loss_s})
    
    m_indep = interpolate(model_a_indep, model_b_indep, th)
    loss_i = eval_loss(m_indep, eval_subset_x, eval_subset_y)
    records.append({'mode': 'Independent initialisation', 'theta': float(th), 'loss': loss_i})

pd.DataFrame(records).to_csv('results/common_init.csv', index=False)
print('Common init finished.')

Common init finished.


Interpolation between models trained from a shared start shows a smooth loss valley at theta=0.5, while independently initialised models hit a high loss barrier.